# Week 5, Day 1: Google ADK - exercise

Seed a different goal on the board, run the worker, then add a fourth tool of my own and watch it in the trace.

In [ ]:
import os, subprocess
from pathlib import Path
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from google.genai import types
from mcp import StdioServerParameters
from google.adk.models.lite_llm import LiteLlm
import board


os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")
load_dotenv(override=True)

MODEL = LiteLlm("openai/gpt-5.4-mini")

## Tools

Three board tools, plus the filesystem MCP server scoped to the `workspace` folder.

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

workspace = Path("task_worker/workspace").resolve()   # the only folder the agent may touch

filesystem = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),
        ),
        timeout=60,
    ),
    errlog=subprocess.DEVNULL,
)

## The worker and the loop

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = LlmAgent(
    model=MODEL,
    name="task_worker",
    instruction=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

async def run_worker(agent, goal):
    board.reset_board()
    goal_id = board.add_goal(goal)
    board.claim_todo(goal_id)

    runner = InMemoryRunner(agent=agent)
    session = await runner.session_service.create_session(app_name=runner.app_name, user_id="ed")

    async for event in runner.run_async(
        user_id="ed",
        session_id=session.id,
        new_message=types.UserContent("Please work the pending goal on the board."),
    ):
        for call in event.get_function_calls():
            print(f"tool: {call.name}({call.args})")
        if any(r.name in ("plan_steps", "complete_task") for r in event.get_function_responses()):
            board.show_board()
        if event.is_final_response():
            print(event.content.parts[0].text)

## Part 1: a different goal

In [ ]:
await run_worker(worker, "Write a short haiku about Madrid into madrid.txt.")
print((workspace / "madrid.txt").read_text())

## Part 2: a fourth tool of my own

In [ ]:
def count_words(text: str) -> dict:
    """Count the words and lines in a piece of text. Use it to check that written content has the length or shape the goal asks for."""
    return {"words": len(text.split()), "lines": len(text.strip().splitlines())}

worker2 = LlmAgent(
    model=MODEL,
    name="task_worker",
    instruction=INSTRUCTIONS + "\nBefore you write a file, use count_words to check the content has the shape the goal asks for.",
    tools=[show_todos, plan_steps, complete_task, count_words, filesystem],
)

await run_worker(worker2, "Write a short haiku about Madrid into madrid.txt. A haiku has exactly 3 lines.")
print((workspace / "madrid.txt").read_text())